# 🧠 Steering DiffuGPT via SAE Feature Intervention

**Built on official [HKUNLP/DiffuLLaMA](https://github.com/HKUNLP/DiffuLLaMA) code**

Extending DLM-Scope (ICLR 2026) with contrastive SAE feature discovery for controllable reasoning.

**Runtime**: `Runtime → Change runtime type → T4 GPU`

In [ ]:
# ========== CELL 1: Setup ==========
!pip install -q transformers==4.44.2 accelerate datasets scipy seaborn safetensors tqdm
import torch
print(f'CUDA: {torch.cuda.is_available()}, GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"}')

In [ ]:
# ========== CELL 2: Official DiffuGPT Model (from HKUNLP/DiffuLLaMA) ==========
# Source: https://github.com/HKUNLP/DiffuLLaMA/blob/main/model.py
# Source: https://github.com/HKUNLP/DiffuLLaMA/blob/main/attention_patch.py

import torch, torch.nn as nn, torch.nn.functional as F
import torch.distributions as dists
from transformers import AutoConfig, AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import PyTorchModelHubMixin
import transformers
from transformers.modeling_attn_mask_utils import _prepare_4d_causal_attention_mask_for_sdpa
from transformers.modeling_outputs import BaseModelOutputWithPastAndCrossAttentions
from typing import Optional, Tuple, Union
import os, json, logging, gc, re
from pathlib import Path

logging.basicConfig(level=logging.INFO)

# --- Monkey-patch GPT2 attention to support 4D mask (from attention_patch.py) ---
orig_gpt2_forward = transformers.models.gpt2.modeling_gpt2.GPT2Model.forward

def patched_gpt2_forward(self, input_ids=None, past_key_values=None, attention_mask=None,
                          token_type_ids=None, position_ids=None, head_mask=None,
                          inputs_embeds=None, encoder_hidden_states=None,
                          encoder_attention_mask=None, use_cache=None,
                          output_attentions=None, output_hidden_states=None, return_dict=None):
    output_attentions = output_attentions if output_attentions is not None else self.config.output_attentions
    output_hidden_states = output_hidden_states if output_hidden_states is not None else self.config.output_hidden_states
    use_cache = use_cache if use_cache is not None else self.config.use_cache
    return_dict = return_dict if return_dict is not None else self.config.use_return_dict

    if input_ids is not None and inputs_embeds is not None:
        raise ValueError('Cannot specify both input_ids and inputs_embeds')
    elif input_ids is not None:
        input_shape = input_ids.size()
        input_ids = input_ids.view(-1, input_shape[-1])
        batch_size = input_ids.shape[0]
    elif inputs_embeds is not None:
        input_shape = inputs_embeds.size()[:-1]
        batch_size = inputs_embeds.shape[0]
    else:
        raise ValueError('Must specify input_ids or inputs_embeds')

    device = input_ids.device if input_ids is not None else inputs_embeds.device
    past_length = 0
    if past_key_values is None:
        past_key_values = tuple([None] * len(self.h))
    if position_ids is None:
        position_ids = torch.arange(0, input_shape[-1], dtype=torch.long, device=device).unsqueeze(0)
    if inputs_embeds is None:
        inputs_embeds = self.wte(input_ids)
    hidden_states = inputs_embeds + self.wpe(position_ids)

    # KEY: Support 4D attention mask from DiffuGPT
    if attention_mask is not None and len(attention_mask.shape) == 4:
        pass  # Use 4D mask as-is
    elif attention_mask is not None:
        attention_mask = attention_mask[:, None, None, :]
        attention_mask = attention_mask.to(dtype=hidden_states.dtype)
        attention_mask = (1.0 - attention_mask) * torch.finfo(hidden_states.dtype).min

    head_mask = self.get_head_mask(head_mask, self.config.n_layer)
    hidden_states = self.drop(hidden_states)
    output_shape = (-1,) + input_shape[1:] + (hidden_states.size(-1),)

    all_hidden_states = () if output_hidden_states else None
    for i, (block, layer_past) in enumerate(zip(self.h, past_key_values)):
        if output_hidden_states:
            all_hidden_states = all_hidden_states + (hidden_states,)
        outputs = block(hidden_states, layer_past=layer_past, attention_mask=attention_mask,
                       head_mask=head_mask[i], use_cache=use_cache, output_attentions=output_attentions)
        hidden_states = outputs[0]

    hidden_states = self.ln_f(hidden_states)
    hidden_states = hidden_states.view(output_shape)
    if output_hidden_states:
        all_hidden_states = all_hidden_states + (hidden_states,)

    if not return_dict:
        return tuple(v for v in [hidden_states, None, all_hidden_states, None, None] if v is not None)
    return BaseModelOutputWithPastAndCrossAttentions(last_hidden_state=hidden_states,
                                                     hidden_states=all_hidden_states)

transformers.models.gpt2.modeling_gpt2.GPT2Model.forward = patched_gpt2_forward
print('✅ GPT2 attention patched for 4D mask support')

# --- Official DiscreteDiffusionModel ---
class DiscreteDiffusionModel(nn.Module, PyTorchModelHubMixin):
    def __init__(self, model, config, tokenizer, device='cuda'):
        super().__init__()
        if isinstance(model, str):
            config_pt = AutoConfig.from_pretrained(model)
            self.model = AutoModelForCausalLM.from_config(config_pt)
        else:
            self.model = model
        self.config = config
        if self.model.get_input_embeddings().weight.size(0) != len(tokenizer):
            self.model.resize_token_embeddings(len(tokenizer), pad_to_multiple_of=2)
        self.vocab_size = self.model.get_input_embeddings().weight.size(0)
        self.embed_tokens = self.model.transformer.wte
        self.denoise_model = self.model.transformer
        for block in self.model.transformer.h:
            block.attn.bias.fill_(True)  # Remove causal mask
        self.lm_head = self.model.lm_head
        del self.denoise_model.wte
        del self.model
        self._device = device

    def get_embeds(self, input_ids): return self.embed_tokens(input_ids)
    def get_logits(self, h): return self.lm_head(h)

    def forward(self, input_ids, attention_mask, inputs_embeds=None, **kwargs):
        x_embed = self.get_embeds(input_ids)
        x = self.denoise_model(inputs_embeds=x_embed, attention_mask=attention_mask, return_dict=False)[0]
        return self.get_logits(x)

def top_p_logits(logits, p=0.9):
    sorted_logits, sorted_indices = torch.sort(logits, descending=True)
    cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
    remove = cumulative_probs > p
    remove[..., 1:] = remove[..., :-1].clone()
    remove[..., 0] = 0
    mask = torch.zeros_like(logits, dtype=torch.bool).scatter_(-1, sorted_indices, remove)
    return logits.masked_fill(mask, torch.finfo(logits.dtype).min)

def get_anneal_attn_mask(seq_len, bsz, dtype, device, ratio=1.0):
    mask = torch.full((seq_len, seq_len), 0, device=device)
    cond = torch.arange(mask.size(-1), device=device)
    mask.masked_fill_(cond < (cond + 1).view(mask.size(-1), 1), 1)
    causal = mask.to(dtype)
    rand = torch.bernoulli(torch.full((seq_len, seq_len), 0.0, device=device) + ratio)
    anneal = torch.logical_or(causal, rand)
    expanded = anneal[None, None, :, :].expand(bsz, 1, seq_len, seq_len)
    inverted = 1.0 - expanded.to(dtype)
    return inverted.masked_fill(inverted.bool(), torch.finfo(dtype).min)

@torch.no_grad()
def generate_samples(model, tokenizer, x, src_mask=None, steps=64,
                     logits_temp=0.95, topp=0.9, shift=True,
                     steering_hook=None, steering_layer=None,
                     collect_acts=False, act_layers=None):
    """Official DiffuGPT generation with optional SAE steering hook."""
    model.eval()
    device = next(model.parameters()).device
    x = x.to(device)
    B, L = x.shape

    if src_mask is None:
        src_mask = torch.zeros_like(x, dtype=torch.bool)
    else:
        src_mask = src_mask.bool().to(device)

    x_embed = model.get_embeds(x)
    attn_mask = get_anneal_attn_mask(L, B, x_embed.dtype, device, 1.0)
    maskable = ~src_mask
    xt = x.masked_fill(maskable, tokenizer.mask_token_id)

    # Activation collection setup
    hooks_list, activations = [], {}
    if collect_acts and act_layers:
        for idx in act_layers:
            block = model.denoise_model.h[idx]
            def mk(i):
                def fn(m, inp, out): activations[i] = (out[0] if isinstance(out, tuple) else out).detach().cpu()
                return fn
            hooks_list.append(block.register_forward_hook(mk(idx)))

    # Steering hook setup
    steer_handle = None
    if steering_hook and steering_layer is not None:
        block = model.denoise_model.h[steering_layer]
        def _steer(mod, inp, out, _h=steering_hook, _m=maskable):
            return (_h(out[0], _m),) + out[1:] if isinstance(out, tuple) else _h(out, _m)
        steer_handle = block.register_forward_hook(_steer)

    # First forward
    logits = model(xt, attention_mask=attn_mask)
    filtered = top_p_logits(logits / logits_temp, p=topp)
    scores = torch.log_softmax(filtered, dim=-1)
    x0 = dists.Categorical(logits=scores).sample()

    if shift:
        x0 = torch.cat([x[:, 0:1], x0[:, :-1]], dim=1)

    x0 = xt.masked_scatter(maskable, x0[maskable])
    mid_acts = None

    # Denoising loop (reverse T-1 to 1)
    for t in range(steps - 1, 0, -1):
        p = 1.0 / (t + 1)
        reveal = maskable & (torch.rand_like(x0, dtype=torch.float) < p)
        xt.masked_scatter_(reveal, x0[reveal])
        maskable = maskable.masked_fill(reveal, False)

        logits = model(xt, attention_mask=attn_mask)
        filtered = top_p_logits(logits / logits_temp, p=topp)
        scores = torch.log_softmax(filtered, dim=-1)
        x0 = dists.Categorical(logits=scores).sample()

        if shift:
            x0 = torch.cat([x[:, 0:1], x0[:, :-1]], dim=1)
        x0 = xt.masked_scatter(maskable, x0[maskable])

        # Capture mid-step activations
        if collect_acts and t == steps // 2:
            mid_acts = {k: v.clone() for k, v in activations.items()}

    if shift:
        x0 = x0[:, 1:]

    # Cleanup
    for h in hooks_list: h.remove()
    if steer_handle: steer_handle.remove()

    return x0, mid_acts

print('✅ Official DiscreteDiffusionModel & generate_samples defined')

In [ ]:
# ========== CELL 3: Top-K SAE ==========
class TopKSAE(nn.Module):
    def __init__(self, d_model, d_dict, k=64):
        super().__init__()
        self.d_model, self.d_dict, self.k = d_model, d_dict, k
        self.encoder = nn.Linear(d_model, d_dict)
        self.decoder = nn.Linear(d_dict, d_model)
        self.pre_bias = nn.Parameter(torch.zeros(d_model))
        with torch.no_grad(): self.decoder.weight.data = F.normalize(self.decoder.weight.data, dim=0)
        self.counts = torch.zeros(d_dict, dtype=torch.long)
    def encode(self, x):
        h = F.relu(self.encoder(x - self.pre_bias))
        v, i = h.topk(self.k, dim=-1)
        out = torch.zeros_like(h); out.scatter_(-1, i, v)
        return out
    def decode(self, h): return self.decoder(h) + self.pre_bias
    def forward(self, x):
        h = self.encode(x); xh = self.decode(h)
        loss = F.mse_loss(xh, x)
        ev = 1 - (x-xh).pow(2).sum() / ((x-x.mean(0)).pow(2).sum()+1e-8)
        self.counts += ((h>0).any(0) if h.dim()==2 else (h>0).any(0).any(0)).long().cpu()
        return xh, h, {'loss': loss, 'ev': ev}
    def get_dirs(self, idx): return self.decoder.weight[:, idx].detach()
    def norm_dec(self):
        with torch.no_grad(): self.decoder.weight.data = F.normalize(self.decoder.weight.data, dim=0)
    def save(self, p):
        p=Path(p); p.mkdir(parents=True,exist_ok=True)
        torch.save(self.state_dict(), p/'w.pt')
        json.dump({'d_model':self.d_model,'d_dict':self.d_dict,'k':self.k}, open(p/'c.json','w'))
print('✅ TopKSAE defined')

In [ ]:
# ========== CELL 4: Load DiffuGPT & Test ==========
SAVE_DIR = '/content/dlm_results'
os.makedirs(SAVE_DIR, exist_ok=True)
cache = os.path.join(SAVE_DIR, 'cache')

model_name = 'diffusionfamily/diffugpt-m'
base_name = 'gpt2-medium'

config = AutoConfig.from_pretrained(model_name, cache_dir=cache)
tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir=cache)

print('Loading DiffuGPT-Medium...')
model = DiscreteDiffusionModel.from_pretrained(
    model_name, model=base_name, config=config,
    tokenizer=tokenizer, device='cuda'
).to('cuda')

d_model = config.hidden_size  # 1024
n_layers = config.n_layer     # 24

print(f'✅ DiffuGPT: d={d_model}, layers={n_layers}')

# Test generation
gen_len = 64
print('\n🧪 Unconditional generation:')
x0 = torch.tensor([[0] * gen_len])
out, _ = generate_samples(model, tokenizer, x0, steps=32)
print(tokenizer.decode(out[0].tolist())[:200])

print('\n🧪 Conditional generation:')
prefix = [tokenizer.bos_token_id] + tokenizer.encode('The capital of France is')
src_mask = [1]*len(prefix) + [0]*(gen_len-len(prefix))
x = prefix + [0]*(gen_len-len(prefix))
out, _ = generate_samples(model, tokenizer, torch.tensor([x]),
                           src_mask=torch.tensor([src_mask]), steps=32)
print(tokenizer.decode(out[0].tolist())[:200])

print('\n✅ Phase 1 complete — model generates correctly!')

In [ ]:
# ========== CELL 5: GSM8K Data ==========
from datasets import load_dataset

COT_T = 'Solve this math problem step by step.\nQuestion: {q}\nSolution:'
DIR_T = 'What is the numerical answer?\nQuestion: {q}\nAnswer:'

ds = load_dataset('openai/gsm8k', 'main', split='test')
problems = list(ds.select(range(200)))

def get_answer(t):
    m = re.search(r'####\s*([\d,\.\-]+)', t)
    return m.group(1).replace(',','') if m else '0'

def make_input(prompt, gen_len=96):
    toks = [tokenizer.bos_token_id] + tokenizer.encode(prompt)
    plen = len(toks)
    x = toks + [0] * (gen_len - plen) if plen < gen_len else toks[:gen_len]
    mask = [1] * min(plen, gen_len) + [0] * max(0, gen_len - plen)
    return torch.tensor([x]), torch.tensor([mask]), min(plen, gen_len)

disc_idx, eval_idx = list(range(100)), list(range(100, 200))
print(f'✅ GSM8K: {len(problems)} problems, disc={len(disc_idx)}, eval={len(eval_idx)}')

In [ ]:
# ========== CELL 6: Collect Activations ==========
from tqdm import tqdm

LAYERS = [4, 8, 12, 16, 20]
N_DISC = 50

def collect(template, indices, n):
    acts = {l: [] for l in LAYERS}
    for i in tqdm(indices[:n], desc='Collecting'):
        p = problems[i]
        prompt = template.format(q=p['question'])
        x, mask, plen = make_input(prompt)
        _, mid = generate_samples(model, tokenizer, x, src_mask=mask, steps=20,
                                   collect_acts=True, act_layers=LAYERS)
        if mid:
            for l in LAYERS:
                if l in mid:
                    acts[l].append(mid[l].float().mean(dim=1))
        if (i+1) % 10 == 0: gc.collect(); torch.cuda.empty_cache()
    return {l: torch.cat(a, 0) for l, a in acts.items() if a}

print('📊 Collecting CoT activations...')
cot_acts = collect(COT_T, disc_idx, N_DISC)
print(f'📊 Collecting Direct activations...')
dir_acts = collect(DIR_T, disc_idx, N_DISC)
torch.save({'cot': cot_acts, 'direct': dir_acts}, f'{SAVE_DIR}/acts.pt')
print(f'✅ Phase 2: shapes = {{", ".join(f"L{l}:{cot_acts[l].shape}" for l in LAYERS if l in cot_acts)}}')

In [ ]:
# ========== CELL 7: Train SAE ==========
from torch.utils.data import DataLoader, TensorDataset
LYR = 16
train_data = torch.cat([cot_acts[LYR], dir_acts[LYR]], 0)
d_dict, K = 4 * d_model, 64
sae = TopKSAE(d_model, d_dict, K).cuda()
opt = torch.optim.Adam(sae.parameters(), lr=3e-4)
loader = DataLoader(TensorDataset(train_data), batch_size=32, shuffle=True)
for ep in range(15):
    tl, te, n = 0, 0, 0
    for (b,) in loader:
        _, _, m = sae(b.cuda()); opt.zero_grad(); m['loss'].backward()
        torch.nn.utils.clip_grad_norm_(sae.parameters(), 1.0)
        opt.step(); sae.norm_dec()
        tl += m['loss'].item(); te += m['ev'].item(); n += 1
    if (ep+1) % 3 == 0: print(f'  Ep {ep+1}: loss={tl/n:.4f}, EV={te/n:.3f}, dead={int((sae.counts==0).sum())}')
sae.eval(); sae.save(f'{SAVE_DIR}/sae')
print(f'✅ Phase 3: SAE trained (EV={te/n:.3f})')

In [ ]:
# ========== CELL 8: Contrastive Feature Discovery ==========
import numpy as np
from scipy import stats
sae.eval()
with torch.no_grad():
    cf = sae.encode(cot_acts[LYR].cuda()).cpu().numpy()
    df = sae.encode(dir_acts[LYR].cuda()).cpu().numpy()
diffs, pvals, cohens = [], [], []
for f in range(d_dict):
    c, d = cf[:, f], df[:, f]; diff = c.mean()-d.mean()
    cs, ds = c.std()+1e-8, d.std()+1e-8
    _, p = stats.ttest_ind(c, d, equal_var=False) if (cs>1e-7 or ds>1e-7) else (0,1.0)
    cd = diff/(np.sqrt((cs**2+ds**2)/2)+1e-8)
    diffs.append(diff); pvals.append(p); cohens.append(cd)
diffs, pvals, cohens = np.array(diffs), np.array(pvals), np.array(cohens)
sig = pvals < (0.05/d_dict)
r_mask = sig & (np.abs(cohens)>=0.2) & (diffs>0)
reasoning_feats = np.where(r_mask)[0][np.argsort(cohens[r_mask])[::-1]].tolist()[:50]
if not reasoning_feats:
    print('⚠️ No Bonferroni-sig features. Using top by effect size.')
    reasoning_feats = [int(i) for i in np.argsort(cohens)[::-1][:50] if diffs[i]>0][:50]
print(f'🔬 {len(reasoning_feats)} reasoning features. Top 10: {reasoning_feats[:10]}')

In [ ]:
# ========== CELL 9: Steering Experiments ==========
import random

def make_steering_hook(sae, feats, alpha):
    d = sae.get_dirs(feats).sum(1)
    d = d / (d.norm() + 1e-8)
    def hook(hs, mask): return hs + alpha * d.to(hs.device).unsqueeze(0).unsqueeze(0)
    return hook

def run_exp(label, feats=None, alpha=0.0, n=20):
    hook = make_steering_hook(sae, feats, alpha) if (alpha != 0 and feats) else None
    layer = LYR if hook else None
    results = []
    for i in tqdm(eval_idx[:n], desc=label):
        p = problems[i]
        prompt = COT_T.format(q=p['question'])
        x, mask, plen = make_input(prompt)
        out, _ = generate_samples(model, tokenizer, x, src_mask=mask, steps=20,
                                   steering_hook=hook, steering_layer=layer)
        text = tokenizer.decode(out[0].tolist())
        results.append({'text': text, 'answer': get_answer(p['answer']), 'prompt': prompt})
    return results

N = 20
print('[E1] Baseline...'); bl = run_exp('baseline', n=N)
print('[E2] Positive α=2...'); pos = run_exp('pos', reasoning_feats, 2.0, N)
print('[E3] Negative α=-2...'); neg = run_exp('neg', reasoning_feats, -2.0, N)
print('[E4] Random control...'); random.seed(42)
rnd = run_exp('rnd', random.sample(range(d_dict), 50), 2.0, N)
print('✅ Phase 5 complete')

In [ ]:
# ========== CELL 10: Evaluation ==========
def rscore(t):
    s = sum(1 for w in ['step','first','then','therefore','because','total','calculate'] if w in t.lower())
    s += len(re.findall(r'[+\-*/=]', t))*0.5 + len(re.findall(r'\d+', t))*0.3
    s += len(re.findall(r'\d+\s*[+\-*/]\s*\d+', t))*2.0
    return s

def extract_num(t):
    m = re.search(r'####\s*([\d,\.]+)', t)
    if m: return m.group(1).replace(',','')
    ns = re.findall(r'-?[\d,]+\.?\d*', t)
    return ns[-1].replace(',','') if ns else None

def evaluate(results, label):
    correct = 0
    for r in results:
        pred = extract_num(r['text'])
        if pred:
            try:
                if abs(float(pred) - float(r['answer'])) < 0.01: correct += 1
            except: pass
    scores = [rscore(r['text']) for r in results]
    return {'label': label, 'acc': correct/len(results), 'rscore': np.mean(scores), 'n': len(results), 'correct': correct}

print('='*60)
print(f'{"Condition":<25} {"Accuracy":<12} {"Reasoning":<12} {"Correct"}')
print('-'*60)
evals = {}
for res, lbl in [(bl,'Baseline'),(pos,'Positive α=2'),(neg,'Negative α=-2'),(rnd,'Random Ctrl')]:
    e = evaluate(res, lbl); evals[lbl] = e
    print(f'{lbl:<25} {e["acc"]:>8.1%}     {e["rscore"]:>8.1f}       {e["correct"]}/{e["n"]}')

print('\n📝 Samples:')
for i in range(min(3, N)):
    print(f'\n--- Q{i+1} (answer: {bl[i]["answer"]}) ---')
    print(f'Base:    {bl[i]["text"][:120]}')
    print(f'Steered: {pos[i]["text"][:120]}')

In [ ]:
# ========== CELL 11: Visualization ==========
import matplotlib.pyplot as plt, seaborn as sns
plt.style.use('dark_background')
C = {'pos':'#00d4aa','neg':'#ff6b6b','rnd':'#ffd93d','base':'#6c8ebf'}
os.makedirs(f'{SAVE_DIR}/figs', exist_ok=True)

fig, (a1,a2) = plt.subplots(1,2,figsize=(14,5))
labs = list(evals.keys()); cols = [C['base'],C['pos'],C['neg'],C['rnd']]
a1.bar(range(4), [evals[l]['acc']*100 for l in labs], color=cols)
a1.set_xticks(range(4)); a1.set_xticklabels(labs, fontsize=9)
a1.set_ylabel('Accuracy %'); a1.set_title('GSM8K Accuracy', fontweight='bold')
a2.bar(range(4), [evals[l]['rscore'] for l in labs], color=cols)
a2.set_xticks(range(4)); a2.set_xticklabels(labs, fontsize=9)
a2.set_ylabel('Score'); a2.set_title('Reasoning Quality', fontweight='bold')
plt.tight_layout(); plt.savefig(f'{SAVE_DIR}/figs/results.png', dpi=150); plt.show()

fig, ax = plt.subplots(figsize=(12,5))
top = reasoning_feats[:25]
ax.barh(range(len(top)), [diffs[f] for f in top],
        color=[plt.cm.viridis(cohens[f]/max(cohens[top]+1e-8)) for f in top])
ax.set_yticks(range(len(top))); ax.set_yticklabels([f'F{f}' for f in top], fontsize=8)
ax.set_xlabel('Differential Activation (CoT − Direct)')
ax.set_title('Top Reasoning SAE Features', fontweight='bold')
ax.invert_yaxis(); plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/figs/features.png', dpi=150); plt.show()

json.dump({'evals':evals, 'features':reasoning_feats,
           'config':{'d_model':d_model,'d_dict':d_dict,'k':K,'layer':LYR}},
          open(f'{SAVE_DIR}/results.json','w'), indent=2, default=str)
print(f'💾 Saved to {SAVE_DIR}/  🏁 Done!')